Đọc dữ liệu

In [2]:

import os
import numpy as np
import pandas as pd

# Đường dẫn dữ liệu giống Decision Tree
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc dữ liệu
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước dữ liệu
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

print("\n--- 5 dòng đầu tiên ---")
print(diamonds.head())

print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
print(f"Cut: {diamonds['cut'].unique()}")
print(f"Color: {diamonds['color'].unique()}")
print(f"Clarity: {diamonds['clarity'].unique()}")

print("\n--- Thống kê mô tả ---")
print(diamonds.describe())


Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [3]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (x, y, z <= 0)
initial_rows = len(diamonds)

diamonds = diamonds[
    (diamonds['x'] > 0) &
    (diamonds['y'] > 0) &
    (diamonds['z'] > 0)
].copy()

invalid_dimensions = initial_rows - len(diamonds)

print(
    f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu "
    "(x, y, z <= 0)."
)

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
num_duplicates = diamonds.duplicated().sum()

diamonds = diamonds.drop_duplicates()

print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 5: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")
print(diamonds.head())



Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [4]:

from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])

# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


Training và kết quả đánh giá

In [5]:

# Import mô hình Random Forest hồi quy
from sklearn.ensemble import RandomForestRegressor

# Import các hàm đánh giá
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import K-Fold
from sklearn.model_selection import KFold

# Import thư viện đo thời gian
import time

# Import thư viện numpy
import numpy as np

# Import thư viện pandas
import pandas as pd

# Tạo mảng lưu trữ kết quả đánh giá cho từng Fold
results = []

# Bắt đầu đo tổng thời gian
start_time = time.perf_counter()

# Duyệt qua từng Fold
for fold, (train_index, test_index) in enumerate(Kf.split(X), start=1):

    # Chia dữ liệu Train và Test
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    # Khởi tạo mô hình Random Forest mặc định
    model = RandomForestRegressor(
        criterion="squared_error",
        random_state=42
    )

    # Bắt đầu đo thời gian huấn luyện
    start_train_time = time.perf_counter()

    # Huấn luyện mô hình trên tập Train
    model.fit(X_train, y_train)

    # Tính thời gian huấn luyện
    training_time = time.perf_counter() - start_train_time

    # Dự đoán trên tập Train
    y_pred_train = model.predict(X_train)

    # Dự đoán trên tập Test
    y_pred_test = model.predict(X_test)

    # Tính các chỉ số Train
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # Tính các chỉ số Test
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    # Lưu kết quả
    results.append({
        "Fold": fold,
        "MAE Train": mae_train,
        "MSE Train": mse_train,
        "RMSE Train": rmse_train,
        "R2 Train": r2_train,
        "MAE Test": mae_test,
        "MSE Test": mse_test,
        "RMSE Test": rmse_test,
        "R2 Test": r2_test,
        "Training Time (s)": training_time
    })

    print(f"Đã hoàn thành Fold {fold}/5")

# Tính tổng thời gian thực hiện
train_time = time.perf_counter() - start_time

# Tạo bảng kết quả từng Fold
results_df = pd.DataFrame(results)

# Tính giá trị trung bình của 5 Fold
mean_result = results_df.drop(columns=["Fold"]).mean()

# Tạo dòng trung bình
mean_row = pd.DataFrame([mean_result])
mean_row.insert(0, "Fold", "Average")

# Ghép kết quả từng Fold và trung bình
final_results = pd.concat(
    [results_df, mean_row],
    ignore_index=True
)

# In bảng kết quả
print("KẾT QUẢ RANDOM FOREST - K-FOLD CROSS VALIDATION")
print(final_results.to_string(index=False, float_format="%.4f"))

print(f"Tổng thời gian train: {train_time:.4f} giây")


Đã hoàn thành Fold 1/5
Đã hoàn thành Fold 2/5
Đã hoàn thành Fold 3/5
Đã hoàn thành Fold 4/5
Đã hoàn thành Fold 5/5
KẾT QUẢ RANDOM FOREST - K-FOLD CROSS VALIDATION
   Fold  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test    MSE Test  RMSE Test  R2 Test  Training Time (s)
      1    99.4531 41813.0678    204.4824    0.9974  267.2045 277560.2606   526.8399   0.9824            12.7397
      2    99.1804 40928.8613    202.3088    0.9974  264.9074 284772.7974   533.6411   0.9817            12.6378
      3    99.5220 40568.1738    201.4154    0.9974  264.3154 290536.5176   539.0144   0.9817            12.4996
      4    99.5734 41539.4890    203.8124    0.9974  267.8643 288898.9018   537.4932   0.9822            12.7342
      5    98.7326 39931.4518    199.8286    0.9975  271.1122 323961.3852   569.1761   0.9797            13.8486
Average    99.2923 40956.2087    202.3695    0.9974  267.0808 293145.9725   541.2329   0.9815            12.8920
Tổng thời gian train: 68.4356 giây
